# 03 · 고급 분석: 패널 고정효과  (모듈 4-A)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/03_panel_fe.ipynb)

> 고급 계량 분석도 AI 도움으로 **양쪽 도구에서** 할 수 있다. 국가×연도 패널의 고정효과를
> Python으로, 그리고 STATA로 — **둘 다 같은 결과**. 도구는 우열이 아니라 **환경**으로 고른다.

In [ ]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv").dropna(subset=["gdp_pc","life_exp"])
df["log_gdp"] = np.log(df["gdp_pc"])
print("국가:", df["economy"].nunique(), "· 연도:", df["year"].nunique(), "· 행:", len(df))

## 문제 — 국가의 고유 특성을 통제하려면?
제도·기후·역사처럼 **잘 변하지 않는 국가 고유 효과**를 빼고, *한 나라 안에서* 소득이 오를 때
기대수명이 어떻게 변하는지 보려면 **고정효과**가 필요하다.

### (1) 통제 안 함 — 단순 회귀(pooled)

In [ ]:
pooled = smf.ols("life_exp ~ log_gdp", data=df).fit()
print("pooled  log_gdp 계수 =", round(pooled.params["log_gdp"], 2))

### (2) Python으로 국가 고정효과 (더미를 다 넣어야 함)

In [ ]:
fe = smf.ols("life_exp ~ log_gdp + C(economy)", data=df).fit()
n = sum(c.startswith("C(economy)") for c in fe.params.index)
print("국가FE  log_gdp 계수 =", round(fe.params["log_gdp"], 2), "(국가효과 흡수로 변화)")
print("추가된 국가 더미 =", n, "개 (국가 고정효과)")

### (3) STATA에선 — 같은 분석
```stata
encode economy, gen(country_id)
xtset country_id year
xtreg life_exp log_gdp, fe vce(cluster country_id)
```
- `xtreg, fe` 가 200여 개 국가 고정효과를 흡수하고, `vce(cluster ...)`로 강건표준오차까지. 위 Python의 `C(economy)`와 **동일한 모형**.
- 코드 → `stata/05_panel_fe.do`  ·  **`xtreg`는 폐쇄망에서 인터넷 없이 실행** ✅

---
✅ **포인트**: 고급 계량 모형도 이제 **양쪽에서** 가능 — AI가 코드를 짜준다.
도구는 *우열*이 아니라 **환경**으로 고른다(폐쇄망 현업은 STATA, 외부망 탐색·확장은 Python).